In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/FC26_20250921.csv")
df.drop(columns=["fifa_version", "fifa_update", "fifa_update_date", "player_face_url", "work_rate", "player_url"], inplace=True)

df_draft = df[df['overall'] >= 75].copy()

faixas = [
        (df_draft['overall'] >= 87),
        (df_draft['overall'] >= 84) & (df_draft['overall'] < 87),
        (df_draft['overall'] >= 80) & (df_draft['overall'] < 84),
        (df_draft['overall'] < 80)
]
    
pesos = [1, 5, 20, 100]
df_draft['weight'] = np.select(faixas, pesos, default=0)

/tmp/ipykernel_5668/2539408828.py:4: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/FC26_20250921.csv")


In [2]:
from draft import generate_draft_pack
from bot import Bot
from eafc_utils import f

print("===========================================\n INICIANDO SIMULADOR DE DRAFT EA FC \n===========================================")

formacao_433 = ['ST', 'LW', 'RW', 'CM', 'CM', 'CM', 'LB', 'CB', 'CB', 'RB', 'GK']
meu_time = []
ids_escolhidos = set()

NUMERO_DE_ROLLOUTS = 1000

meu_bot = Bot(mode="expectimax", num_rollouts=NUMERO_DE_ROLLOUTS)

for rodada, posicao_alvo in enumerate(formacao_433, start=1):
    print(f"\n--- RODADA {rodada}/11 | Buscando: {posicao_alvo} ---")

    pacote = generate_draft_pack(df_draft, rodada, posicao_alvo, ids_escolhidos)
    opcoes = pacote.reset_index(drop=True)
    opcoes['choosen_position'] = posicao_alvo
    print(opcoes[['short_name', 'overall', 'club_name', 'league_name', 'nationality_name']])

    df_meu_time = pd.DataFrame(meu_time) if meu_time else pd.DataFrame(columns=df_draft.columns)

    posicoes_restantes = formacao_433[rodada:]

    indice_escolhido = meu_bot.choose(
        options=opcoes,
        current_squad=df_meu_time,
        db=df_draft,
        current_round=rodada,
        remaining_positions=posicoes_restantes
    )

    carta_escolhida = opcoes.iloc[indice_escolhido]
    meu_time.append(carta_escolhida)
    ids_escolhidos.add(carta_escolhida['player_id'])

    print(f">>> ESCOLHA DO BOT: {carta_escolhida['short_name']} (OVR {carta_escolhida['overall']})")

print("\n===========================================")
print(" DRAFT CONCLUÍDO! SEU ELENCO FINAL: ")
print("===========================================")
df_final = pd.DataFrame(meu_time)
print(df_final[['short_name', 'overall', 'league_name', 'nationality_name', 'club_name']])
print(f"\nNOTA FINAL DO TIME: {f(df_final):.2f}")

 INICIANDO SIMULADOR DE DRAFT EA FC 

--- RODADA 1/11 | Buscando: ST ---
    short_name  overall          club_name     league_name nationality_name
0      Alisson       89          Liverpool  Premier League           Brazil
1   J. Kimmich       89  FC Bayern München      Bundesliga          Germany
2  F. Valverde       89        Real Madrid         La Liga          Uruguay
3      B. Saka       88            Arsenal  Premier League          England
4     Raphinha       89       FC Barcelona         La Liga           Brazil
>>> ESCOLHA DO BOT: Alisson (OVR 89)

--- RODADA 2/11 | Buscando: LW ---
    short_name  overall            club_name          league_name  \
0       Rômulo       76           RB Leipzig           Bundesliga   
1  Diego López       78          Valencia CF              La Liga   
2    J. Bahoya       75  Eintracht Frankfurt           Bundesliga   
3     M. Uzuni       75            Austin FC  Major League Soccer   
4    B. Traoré       75        FC Basel 1893         